# Live demo: Агрегатор ↔ Эксплуатант

Этот notebook имитирует работу **Агрегатора**, общаясь с **запущенным** Эксплуатантом (Operator) через брокер сообщений.

- Поддерживаемые транспорты: **Mosquitto/MQTT** и **Kafka** (переключается в ноутбуке).
- Тип взаимодействия: request/response по `SystemBus`.

## Предусловия

Поднять Эксплуатанта в Docker:

- MQTT (default):

```bash
docker compose -f systems/operator/docker-compose.yml up -d --build
```

- Kafka (опционально):

```bash
cd systems/operator && make up-kafka
```

## Сценарий

1. Агрегатор отправляет заказ (`receive_order`), получает предложение (цена/срок/параметры)
2. Агрегатор подтверждает выбор Эксплуатанта (`submit_proposal`, затем `accept_order`)
3. Запуск выполнения (`start_mission`) и проверка статуса (`get_mission_status`)
4. Завершение (`complete_mission`) и финальный статус


## Диаграммы взаимодействия

### Диаграмма 1: жизненный цикл заказа

```mermaid
sequenceDiagram
  participant Customer
  participant Aggregator
  participant Operator

  Customer->>Aggregator: CreateOrder
  Aggregator->>Operator: receive_order(order)
  Operator-->>Aggregator: proposal(price,delivery_time,...)

  Aggregator->>Operator: submit_proposal(order_id)
  Operator-->>Aggregator: submitted

  Aggregator->>Operator: accept_order(order_id)
  Operator-->>Aggregator: accepted(mission_id,uas_id)

  Aggregator->>Operator: start_mission(mission_id)
  Operator-->>Aggregator: started

  loop UntilCompleted
    Aggregator->>Operator: get_mission_status(mission_id)
    Operator-->>Aggregator: status
  end

  Aggregator->>Operator: complete_mission(mission_id)
  Operator-->>Aggregator: completed
```

### Диаграмма 2: envelope request/response

```mermaid
sequenceDiagram
  participant Aggregator
  participant Bus
  participant Operator

  Aggregator->>Bus: request(topic, {action,payload,sender})
  Bus->>Operator: publish(topic, message+{correlation_id,reply_to})
  Operator->>Bus: publish(reply_to, response)
  Bus-->>Aggregator: response({correlation_id,payload,success,error?})
```


### PNG (PlantUML) диаграммы

![](assets/plantuml/operator_components_updated.png)

![](assets/plantuml/uas_purchase_sequence.png)

![](assets/plantuml/fleet_manager_architecture.png)

![](assets/plantuml/uas_reservation_sequence.png)

## Настройки запуска

Следующие кодовые ячейки:

- выбирают брокер (`mqtt` или `kafka`) и формируют топик Operator;
- поднимают нужные контейнеры Эксплуатанта через `make`/`docker compose`;
- готовят `SystemBus` и хелперы для request/response.

In [1]:
# Настройки (выберите брокер: 'mqtt' или 'kafka')
import os
import sys
from pathlib import Path

# В VSCode notebook обычно cwd=notebooks/.
# Добавляем корень репозитория в sys.path, чтобы работали импорты `broker`, `systems`, `sdk`.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "broker").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

broker = os.getenv("AGG_BROKER", "kafka")  # "mqtt" | "kafka"

SYSTEM_ID = os.getenv("SYSTEM_ID", "operator-001")
API_VERSION = os.getenv("API_VERSION", "v1")

# MQTT
MQTT_BROKER = os.getenv("MQTT_BROKER", "localhost")
MQTT_PORT = int(os.getenv("MQTT_PORT", "1883"))

# Kafka (для ноутбука используем внешний listener)
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:19092")

OPERATOR_TOPIC = f"{SYSTEM_ID}.{API_VERSION}.operator"

print("repo_root=", repo_root)
print("broker=", broker)
print("OPERATOR_TOPIC=", OPERATOR_TOPIC)
print("MQTT=", f"{MQTT_BROKER}:{MQTT_PORT}")
print("KAFKA=", KAFKA_BOOTSTRAP_SERVERS)


repo_root= /home/user/projects/sbd-drones-economics/sbd-drones-economics-ai
broker= kafka
OPERATOR_TOPIC= operator-001.v1.operator
MQTT= localhost:1883
KAFKA= localhost:19092


## 1) Инициализация брокера и bus

Код ниже создаёт/поднимает `SystemBus` в выбранном транспорте и готовит reply-топик для ответов от Эксплуатанта.

In [2]:
import time
from dataclasses import dataclass


@dataclass
class BusConfig:
    broker: str
    bus: object


def create_bus(selected: str) -> BusConfig:
    selected = selected.lower().strip()

    if selected == "mqtt":
        from broker.mqtt.mqtt_system_bus import MQTTSystemBus

        b = MQTTSystemBus(broker=MQTT_BROKER, port=MQTT_PORT, client_id=f"aggregator.{SYSTEM_ID}")
        # Не стартуем шину сразу: контейнеры/брокер поднимаются позже.
        return BusConfig(broker="mqtt", bus=b)

    if selected == "kafka":
        from broker.kafka.kafka_system_bus import KafkaSystemBus

        b = KafkaSystemBus(
            bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
            client_id=f"aggregator.{SYSTEM_ID}",
            group_id=f"aggregator-{SYSTEM_ID}",
        )
        # Не стартуем шину сразу: контейнеры/брокер поднимаются позже.
        return BusConfig(broker="kafka", bus=b)

    raise ValueError(f"Unknown broker: {selected}")


bus_cfg = create_bus(broker)
bus = bus_cfg.bus
print("Bus instantiated (will start after broker readiness):", bus_cfg.broker)


Bus instantiated (will start after broker readiness): kafka


## 2) Поднятие контейнеров Operator

Эта ячейка запускает нужный `docker-compose` стек (MQTT или Kafka) через `make`.

Если в вашем окружении `docker buildx` падает с ошибкой `permission denied`, ноутбук автоматически сделает fallback: поднимет стек без `--build`. Если после этого брокер всё равно недоступен с хоста, сценарий будет пропущен вместо того, чтобы завершить notebook ошибкой.

Проверка для исправления проблем:
- MQTT: убедитесь, что доступен `localhost:1883` (или переопределите `MQTT_BROKER/MQTT_PORT`).
- Kafka: убедитесь, что доступен `KAFKA_BOOTSTRAP_SERVERS`.

In [3]:
# Поднять нужный docker-compose стек (через Makefile)
# Важно: требуется установленный Docker.

import subprocess

operator_dir = repo_root / "systems" / "operator"
stack_up_ok = False

if broker == "mqtt":
    cmd = ["make", "-C", str(operator_dir), "up-mqtt"]
    compose_file = "docker-compose.yml"
elif broker == "kafka":
    cmd = ["make", "-C", str(operator_dir), "up-kafka"]
    compose_file = "docker-compose.kafka.yml"
else:
    raise ValueError(f"Unknown broker: {broker}")

print("Running:", " ".join(cmd))

try:
    subprocess.run(cmd, check=True, cwd=str(operator_dir))
    stack_up_ok = True
except subprocess.CalledProcessError as e:
    # В некоторых окружениях Docker buildx может падать из-за permission denied.
    # Тогда пробуем поднять стек без сборки (если образы уже есть).
    print("WARN: make up failed; fallback to docker compose without build")
    print("  error:", e)

    try:
        fallback = [
            "docker",
            "compose",
            "-f",
            compose_file,
            "up",
            "-d",
            "--remove-orphans",
        ]
        subprocess.run(fallback, check=True, cwd=str(operator_dir))
        stack_up_ok = True
    except Exception as e2:  # noqa: BLE001
        print("WARN: fallback also failed; skipping docker demo")
        print("  error:", e2)
except Exception as e:  # noqa: BLE001
    print("WARN: unexpected error while starting stack; skipping docker demo")
    print("  error:", e)


Running: make -C /home/user/projects/sbd-drones-economics/sbd-drones-economics-ai/systems/operator up-kafka
make: Entering directory '/home/user/projects/sbd-drones-economics/sbd-drones-economics-ai/systems/operator'
docker build -t operator-base-kafka:latest -f ../../systems/operator/docker/Dockerfile.base ../.. --build-arg INSTALL_KAFKA=1; \
docker compose -f docker-compose.kafka.yml up -d --build --remove-orphans
make: Leaving directory '/home/user/projects/sbd-drones-economics/sbd-drones-economics-ai/systems/operator'
WARN: make up failed; fallback to docker compose without build
  error: Command '['make', '-C', '/home/user/projects/sbd-drones-economics/sbd-drones-economics-ai/systems/operator', 'up-kafka']' returned non-zero exit status 2.


ERROR: failed to build: failed to update builder last activity time: open /home/user/.docker/buildx/activity/.tmp-default3564479087: permission denied
make: *** [Makefile:80: up-kafka] Error 1
 Container operator-zookeeper Running 
 Container operator-kafka Running 
 Container operator-security-monitor Running 
 Container operator-business-logic Running 
 Container operator-event-journal Running 
 Container operator-mission-planner Running 
 Container operator-fleet-manager Running 
 Container operator-system Running 


## 3) Ожидание healthcheck контейнеров

Здесь ждём, пока компоненты Эксплуатанта перейдут в `healthy/running`, иначе request/response будет падать таймаутами.

In [4]:
# Ожидание готовности контейнеров (healthcheck/running)

import time


def _docker_inspect(fmt: str, name: str) -> str:
    # Returns empty string on error.
    p = subprocess.run(
        ["docker", "inspect", "-f", fmt, name],
        capture_output=True,
        text=True,
    )
    if p.returncode != 0:
        return ""
    return (p.stdout or "").strip()


def wait_container(name: str, timeout_s: int = 180) -> None:
    deadline = time.time() + timeout_s
    last = None
    while time.time() < deadline:
        health = _docker_inspect("{{.State.Health.Status}}", name)
        state = _docker_inspect("{{.State.Status}}", name)
        status = health or state
        if status and status in {"healthy", "running"}:
            print(f"OK: {name} status={status}")
            return
        last = (health, state)
        time.sleep(2)
    raise TimeoutError(f"Timeout waiting {name}: health/state={last}")


mqtt_containers = [
    "operator-mosquitto",
    "operator-security-monitor",
    "operator-fleet-manager",
    "operator-mission-planner",
    "operator-business-logic",
    "operator-system",
]

kafka_containers = [
    "operator-kafka",
    "operator-security-monitor",
    "operator-fleet-manager",
    "operator-mission-planner",
    "operator-business-logic",
    "operator-system",
]

containers = mqtt_containers if broker == "mqtt" else kafka_containers

if not globals().get("stack_up_ok", False):
    print("SKIP: stack_up_ok=False; containers healthcheck skipped.")
else:
    print("Waiting containers:", containers)
    for c in containers:
        wait_container(c, timeout_s=240)


Waiting containers: ['operator-kafka', 'operator-security-monitor', 'operator-fleet-manager', 'operator-mission-planner', 'operator-business-logic', 'operator-system']
OK: operator-kafka status=healthy
OK: operator-security-monitor status=healthy
OK: operator-fleet-manager status=healthy


OK: operator-mission-planner status=healthy


OK: operator-business-logic status=healthy
OK: operator-system status=healthy


## 4) Fleet Manager helpers

Кодовые хелперы для запросов к `fleet_manager` (каталоги разработчиков, покупка UAS) вынесены в отдельную функцию, чтобы не смешивать с запросами к Operator.

In [5]:
# Вспомогательный запрос к Fleet Manager (каталоги, покупка UAS)

def fleet_request(action: str, payload: dict, timeout_s: float = 30.0) -> dict:
    action = action.lower()
    fm_topic = f"{SYSTEM_ID}.fleet_manager"
    resp = bus.request(
        fm_topic,
        {
            "action": action,
            "sender": "aggregator-demo",
            "payload": payload,
        },
        timeout=timeout_s,
    )
    if resp is None:
        return {"error": f"no response from {fm_topic} for {action}"}
    return resp.get("payload", {})


## 5) Readiness проверки брокера

Перед выполнением сценария дополнительно проверяем, что брокер реально доступен (можно подключиться).

Если из текущего окружения брокер недоступен с хоста (например, `ConnectionRefused` для MQTT), ячейка не падает: выставляет `stack_up_ok=False`, и сценарий дальше пропускается.

Проверка для исправления проблем:
- убедитесь, что нужные порты проброшены наружу (для MQTT `1883`, для Kafka `19092`/аналог);
- при необходимости поправьте параметры в начале ноутбука.

In [6]:
# Проверка доступности брокера (readiness)

import time


def wait_broker_ready(timeout_s: float = 30.0) -> None:
    deadline = time.time() + timeout_s
    last_exc: Exception | None = None

    while time.time() < deadline:
        try:
            if broker == "mqtt":
                from broker.mqtt.mqtt_system_bus import MQTTSystemBus

                b = MQTTSystemBus(broker=MQTT_BROKER, port=MQTT_PORT, client_id=f"nb-ready.{SYSTEM_ID}")
                b.start()
                b.stop()
                print("OK: MQTT broker ready")
                return

            if broker == "kafka":
                from broker.kafka.kafka_system_bus import KafkaSystemBus

                b = KafkaSystemBus(
                    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
                    client_id=f"nb-ready.{SYSTEM_ID}",
                    group_id=f"nb-ready-{SYSTEM_ID}",
                )
                b.start()
                b.stop()
                print("OK: Kafka broker ready")
                return

            raise ValueError(f"Unknown broker: {broker}")

        except Exception as e:  # noqa: BLE001
            last_exc = e
            time.sleep(0.5)

    raise TimeoutError(f"Broker not ready after {timeout_s}s: {last_exc}")


if not globals().get("stack_up_ok", False):
    print("SKIP: stack_up_ok=False; not starting main SystemBus.")
else:
    try:
        wait_broker_ready(45.0)
    except TimeoutError as e:
        print("WARN: broker not reachable from this environment; skipping demo")
        print("  error:", e)
        stack_up_ok = False

    if stack_up_ok:
        print("Starting main SystemBus...")
        bus.start()
        print("Bus started:", broker)



WARN: broker not reachable from this environment; skipping demo
  error: Broker not ready after 45.0s: NoBrokersAvailable


## 6) Helper request(action, payload)

Эта функция отправляет request/response в топик `operator` с ретраями только по таймаутам. Бизнес-ошибки возвращаются наверх без повторов.

In [ ]:
def request(action: str, payload: dict, timeout_s: float = 30.0, *, retries: int = 8) -> dict:
    """Aggregator -> Operator request/response with retry.

    Ретрай только по таймаутам брокера/сервиса; бизнес-ошибки возвращаем вызывающему коду.
    Все action приводим к нижнему регистру.
    """

    last_exc: Exception | None = None
    action = action.lower()

    for attempt in range(1, retries + 1):
        try:
            resp = bus.request(
                OPERATOR_TOPIC,
                {
                    "action": action,
                    "sender": "aggregator-demo",
                    "payload": payload,
                },
                timeout=timeout_s,
            )
            if resp is None:
                raise TimeoutError(f"No response for action={action}")

            payload_resp = resp.get("payload", {})
            # Бизнес-ошибки не считаем поводом для retry: просто возвращаем payload наверх.
            return payload_resp

        except TimeoutError as e:
            last_exc = e
            if attempt == retries:
                break
            sleep_s = min(8.0, 0.5 * (2 ** (attempt - 1)))
            print(
                f"WARN: timeout for action={action} attempt={attempt}/{retries}: {e}. "
                f"Retry in {sleep_s:.1f}s"
            )
            time.sleep(sleep_s)

        except Exception as e:  # noqa: BLE001
            last_exc = e
            break

    print("Diagnostics:")
    print("  broker=", broker)
    print("  OPERATOR_TOPIC=", OPERATOR_TOPIC)
    if broker == "mqtt":
        print("  MQTT=", f"{MQTT_BROKER}:{MQTT_PORT}")
    else:
        print("  KAFKA=", KAFKA_BOOTSTRAP_SERVERS)
    return {"error": str(last_exc) if last_exc else "request failed"}


def pretty(d: dict) -> None:
    import json

    print(json.dumps(d, ensure_ascii=False, indent=2))


## 7) Шаг 1 сценария: receive_order (+ покупка UAS при пустом парке)

Сначала создаём заказ и отправляем `receive_order`. Если Эксплуатант вернёт ошибку про отсутствие UAS — покупаем один через `fleet_manager` и повторяем запрос.

In [ ]:
# 1) Создаём заказ и получаем предложение
# Если парк пустой — автоматически покупаем 1 БАС и повторяем запрос.

if not globals().get("stack_up_ok", False):
    print("SKIP step1: operator stack not started (stack_up_ok=False)")
    order_id = None
    proposal = {}
else:

    def ensure_fleet_has_uas() -> None:
        # все действия отправляем в нижнем регистре
        # Fleet-операции выполняем через FleetManager topic
        lst = fleet_request("get_uas_list", {}, timeout_s=30.0)
        if isinstance(lst, dict) and (lst.get("error") or lst.get("success") is False):
            print("WARN: get_uas_list error:", lst)

        total = 0
        if isinstance(lst, dict):
            total = lst.get("total") or lst.get("total_count") or 0

        if total:
            return

        def _fallback_candidates() -> list[tuple[str, str]]:
            # Фолбэк: берём модели из YAML, который использует DeveloperClient в демо/тест-режиме.
            try:
                import yaml

                yaml_path = (
                    repo_root
                    / "systems"
                    / "operator"
                    / "src"
                    / "operator_clients"
                    / "resources"
                    / "developers_catalog.yaml"
                )
                data = yaml.safe_load(yaml_path.read_text(encoding="utf-8")) or {}
                developers = data.get("developers") or []

                candidates: list[tuple[str, str]] = []
                for dev in developers:
                    dev_id = dev.get("developer_id")
                    for m in dev.get("models", []) or []:
                        if int(m.get("available_quantity", 0) or 0) > 0:
                            model_id = m.get("model_id")
                            if dev_id and model_id:
                                candidates.append((str(dev_id), str(model_id)))
                return candidates
            except Exception as e:  # noqa: BLE001
                print("WARN: yaml fallback failed:", type(e).__name__, e)
                return []

        catalogs = fleet_request("get_developer_catalogs", {}, timeout_s=40.0)

        candidates: list[tuple[str, str]] = []
        if isinstance(catalogs, dict) and not catalogs.get("error"):
            cats = catalogs.get("catalogs") or {}
            if cats:
                for dev_id, cat in cats.items():
                    for m in cat.get("models", []) or []:
                        if int(m.get("available_quantity", 0) or 0) > 0:
                            model_id = m.get("model_id")
                            if model_id:
                                candidates.append((str(dev_id), str(model_id)))

        if not candidates:
            candidates = _fallback_candidates()

        if not candidates:
            raise RuntimeError(f"No available UAS candidates (catalogs={catalogs})")

        last_err: dict | None = None
        # Пробуем несколько вариантов, чтобы уменьшить вероятность таймаут/гонку.
        for pick_dev, pick_model in candidates[:10]:
            pur = fleet_request(
                "purchase_uas",
                {"developer_id": pick_dev, "model_id": pick_model, "quantity": 1},
                timeout_s=60.0,
            )
            if isinstance(pur, dict) and pur.get("success") is True:
                return
            last_err = pur if isinstance(pur, dict) else {"error": str(pur)}

        raise RuntimeError(f"PURCHASE_UAS failed for all candidates. last_err={last_err}")

    order_id = f"ORDER-LIVE-{int(time.time())}"
    order = {
        "id": order_id,
        "pickup": {"lat": 55.76, "lon": 37.62},
        "dropoff": {"lat": 55.75, "lon": 37.61},
        "payload_weight": 1.0,
        "distance_km": 2.0,
        "payload_value": 20000,
    }

    # Aggregator явно задаёт свою роль для SecurityMonitor
    receive_res = request(
        "receive_order",
        {"order": order, "sender_role": "aggregator"},
        timeout_s=30.0,
    )
    if receive_res.get("proposal", {}).get("error") == "No suitable UAS available":
        print("Fleet empty/no suitable UAS. Purchasing one and retrying...")
        ensure_fleet_has_uas()
        receive_res = request("receive_order", {"order": order}, timeout_s=30.0)

    print("receive_order response")
    pretty(receive_res)

    proposal = receive_res.get("proposal", {})
    print("proposal")
    pretty(proposal)


SKIP step1: operator stack not started (stack_up_ok=False)


## 8) Шаг 2 сценария: submit_proposal → accept_order

Далее агрегатор подтверждает предложение и выбирает Эксплуатанта исполнителем. На выходе должен появиться `mission_id`.

In [ ]:
# 2) Подтверждаем, что предложение отправлено (submit_proposal)
if not globals().get("stack_up_ok", False):
    print("SKIP step2: operator stack not started (stack_up_ok=False)")
else:
    if "error" in proposal:
        raise RuntimeError(f"Proposal has error, cannot continue: {proposal}")

    submit_res = request("submit_proposal", {"order_id": order_id}, timeout_s=15.0)
    print("submit_proposal response")
    pretty(submit_res)

    if "error" in submit_res:
        raise RuntimeError(f"submit_proposal failed: {submit_res}")

    # 3) Выбираем Эксплуатанта исполнителем (accept_order)
    accept_res = request("accept_order", {"order_id": order_id}, timeout_s=30.0)
    print("accept_order response")
    pretty(accept_res)

    if "error" in accept_res:
        raise RuntimeError(f"accept_order failed: {accept_res}")

    mission_id = accept_res.get("mission_id")
    if not mission_id:
        raise RuntimeError(f"accept_order did not return mission_id: {accept_res}")


SKIP step2: operator stack not started (stack_up_ok=False)


## 9) Шаг 3 сценария: start_mission → complete_mission

Запускаем выполнение миссии, опрашиваем статус и завершаем заказ.

In [ ]:
# 4) Запускаем выполнение и опрашиваем статус
if not globals().get("stack_up_ok", False):
    print("SKIP step3: operator stack not started (stack_up_ok=False)")
else:
    start_res = request("start_mission", {"mission_id": mission_id}, timeout_s=30.0)
    print("start_mission response")
    pretty(start_res)

    status = None
    for _ in range(10):
        status = request("get_mission_status", {"mission_id": mission_id}, timeout_s=15.0)
        print("get_mission_status")
        pretty(status)
        if status.get("status") in {"completed", "failed", "aborted"}:
            break
        time.sleep(1.0)

    # 5) Завершаем миссию (если ещё не завершена)
    if status and status.get("status") != "completed":
        complete_res = request("complete_mission", {"mission_id": mission_id}, timeout_s=30.0)
        print("complete_mission response")
        pretty(complete_res)

    final_status = request("get_mission_status", {"mission_id": mission_id}, timeout_s=15.0)
    print("final get_mission_status")
    pretty(final_status)


SKIP step3: operator stack not started (stack_up_ok=False)


## 10) Cleanup

Отключаем reply-каналы и (по желанию) останавливаем `SystemBus`.

In [ ]:
# Cleanup (по желанию)
# bus.stop()
# print("Bus stopped")
